In [ ]:
!pip install -q --upgrade transformers tokenizers datasets librosa soundfile accelerate jiwer


In [ ]:
!pip uninstall -y transformers
!pip install -U transformers==4.44.2


In [ ]:
import os
import pandas as pd
import librosa
from tqdm import tqdm
from datasets import Dataset, DatasetDict
from transformers import WhisperProcessor

TSV_PATH = "/kaggle/input/urdu-dataset-20000/final_main_dataset.tsv"
AUDIO_DIR = "/kaggle/input/urdu-dataset-20000/limited_wav_files/limited_wav_files"
SAVE_DIR = "/kaggle/working/urdu_dataset_prepared"
SAMPLE_N = 9000


In [ ]:
df = pd.read_csv(TSV_PATH, sep="\t")
print("Loaded TSV with", len(df), "rows")

def resolve_path(rel_path):
    base_name = os.path.splitext(os.path.basename(rel_path))[0] + ".wav"
    full_path = os.path.join(AUDIO_DIR, base_name)
    return full_path if os.path.exists(full_path) else None

df["audio_path"] = df["path"].apply(resolve_path)
df = df[df["audio_path"].notnull()]
print("Valid audio files found:", len(df))
print(df[["path", "audio_path"]].head())

if SAMPLE_N and len(df) > SAMPLE_N:
    df = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
    print(f"Using sample of {SAMPLE_N} rows for speed")


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
print("Split dataset into train/test")

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="ur", task="transcribe")


In [ ]:
def preprocess(batch):
    input_features, labels = [], []

    for path, text in zip(batch["audio_path"], batch["sentence"]):
        y, _ = librosa.load(path, sr=16000, mono=True)
        inputs = processor(y, sampling_rate=16000, return_tensors="pt")
        input_features.append(inputs.input_features[0])
        labels.append(processor.tokenizer(text, truncation=True).input_ids)

    return {"input_features": input_features, "labels": labels}


In [ ]:
prepared = {
    "train": dataset_split["train"].map(
        preprocess,
        batched=True,
        batch_size=8,
        num_proc=1,
        remove_columns=dataset_split["train"].column_names
    ),
    "test": dataset_split["test"].map(
        preprocess,
        batched=True,
        batch_size=8,
        num_proc=1,
        remove_columns=dataset_split["test"].column_names
    ),
}

prepared = DatasetDict(prepared)
prepared.save_to_disk(SAVE_DIR)
print(f"Preprocessing complete and saved to: {SAVE_DIR}")
